# Validation Notebook

## Evaluation Metric: Accuracy

We evaluate extraction quality using **Accuracy** — the fraction of rows where the predicted value exactly matches the gold standard value (after normalisation).

$$\text{Accuracy} = \frac{\text{Correct}}{N}$$

Where:
- **Correct** = number of rows where `norm(pred) == norm(gold)`
- **N** = total number of rows

Normalisation lowercases and strips values; empty strings, `"nan"`, and `"none"` are mapped to `"__none__"`. Two `"__none__"` values count as a match (both correctly absent).

Mismatches are listed as errors showing the predicted vs gold value for each row.

In [9]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.join(os.path.abspath('.'), 'Target-Event-Agent_Networks'))
from teanets.nlp_utils import get_spacy_nlp
from teanets.svo_extraction import extract_svos

nlp = get_spacy_nlp()

def norm(x):
    s = str(x).strip().lower()
    return "__none__" if s in {"", "nan", "none"} else s

def accuracy(pred, gold):
    p = pred.apply(norm)
    g = gold.apply(norm)
    correct = (p == g).sum()
    total = len(p)
    return {"Correct": correct, "Total": total, "Accuracy": round(correct / total, 3) if total else 0.0}

In [10]:
CORPUS_FILE = "../data/sample_sentences.csv"

df_corpus = pd.read_csv(CORPUS_FILE)
if "sentence" not in df_corpus.columns:
    raise ValueError("Corpus miss 'sentence' column.")

wdw_rows = []
for i, s in enumerate(df_corpus["sentence"].fillna("")):
    df_rel = extract_svos(nlp(s))

    who_did = df_rel[(df_rel["TEA"] == "Agent") & (df_rel["TEA2"] == "Event")] if not df_rel.empty else pd.DataFrame()
    did_what = df_rel[(df_rel["TEA"] == "Event") & (df_rel["TEA2"] == "Target")] if not df_rel.empty else pd.DataFrame()

    who = did = what = None
    if not who_did.empty:
        who = who_did.iloc[0]["Node 1"]
        did = who_did.iloc[0]["Node 2"]
    if not did_what.empty:
        if did is None:
            did = did_what.iloc[0]["Node 1"]
        same_verb = did_what[did_what["Node 1"] == did]
        what = same_verb.iloc[0]["Node 2"] if not same_verb.empty else did_what.iloc[0]["Node 2"]

    wdw_rows.append({
        "sent_id": i,
        "sentence": s,
        "agent": who,
        "event": did,
        "target": what,
    })

df_tea = pd.DataFrame(wdw_rows)
print(f"Corpus' sentences: {len(df_corpus)}")
print(f"Extracted Triplets (rows): {len(df_tea)}")
df_tea[["sentence", "agent", "event", "target"]].head(50)

Corpus' sentences: 122
Extracted Triplets (rows): 122


,sentence,agent,event,target
0,The cat chased the mouse.,cat,chase,mouse
1,John reads books.,john,read,book
2,Mary loves chocolate.,mary,love,chocolate
3,The dog barks loudly.,dog,bark loudly,NaN
4,The sun sets in the west.,sun,set,in west
5,Birds fly.,bird,fly,NaN
6,The student is reading a book.,student,read,book
7,They have finished their homework.,they,finish,their homework
8,He will eat the apple.,he,will eat,apple
9,The team had lost the game.,team,lose,game


In [12]:
GOLD_FILE = "../data/gold_standard_svo.csv"
df_gold = pd.read_csv(GOLD_FILE)

def extract_svo_dep(doc):
    """Baseline: pure dependency-based SVO, then map to semantic roles."""
    subj = verb = obj = None
    is_passive = False
    for token in doc:
        if token.dep_ == 'ROOT':
            verb = token.lemma_
            for child in token.children:
                if child.dep_ in ('nsubj', 'nsubjpass', 'csubj', 'csubjpass'):
                    subj = child.lemma_
                    if child.dep_ in ('nsubjpass', 'csubjpass'):
                        is_passive = True
                if child.dep_ in ('dobj', 'pobj', 'attr', 'acomp', 'ccomp'):
                    obj = child.lemma_
                elif child.dep_ == 'prep':
                    for p_child in child.children:
                        if p_child.dep_ == 'pobj':
                            obj = f"{child.lemma_} {p_child.lemma_}"
                elif child.dep_ == 'agent':
                    for p_child in child.children:
                        if p_child.dep_ == 'pobj':
                            obj = p_child.lemma_
                            is_passive = True
                elif child.dep_ == 'xcomp':
                    for xc_child in child.children:
                        if xc_child.dep_ in ('dobj', 'pobj', 'attr'):
                            obj = xc_child.lemma_
            break
    # Map syntactic → semantic roles
    if is_passive and obj:
        return obj, verb, subj
    elif is_passive:
        return subj, verb, None
    else:
        return subj, verb, obj

gold_rows = []
for i, row in df_gold.iterrows():
    s = str(row["sentence"])
    if s == 'nan': continue
    doc = nlp(s)
    agent, event, target = extract_svo_dep(doc)
    gold_rows.append({
        "sent_id": row.get("sent_id", i),
        "sentence": row["sentence"],
        "pred_agent": agent,
        "pred_event": event,
        "pred_target": target,
        "gold_agent": row["agent"],
        "gold_event": row["event"],
        "gold_target": row["target"],
    })

df_eval = pd.DataFrame(gold_rows)
metrics_baseline = {
    "Agent":  accuracy(df_eval["pred_agent"],  df_eval["gold_agent"]),
    "Event":  accuracy(df_eval["pred_event"],  df_eval["gold_event"]),
    "Target": accuracy(df_eval["pred_target"], df_eval["gold_target"]),
}

# TEA Nets extraction
tea_rows = []
for i, row in df_gold.iterrows():
    s = str(row["sentence"])
    if s == 'nan': continue
    df_rel = extract_svos(nlp(s))

    who_did = df_rel[(df_rel["TEA"] == "Agent") & (df_rel["TEA2"] == "Event")] if not df_rel.empty else pd.DataFrame()
    did_what = df_rel[(df_rel["TEA"] == "Event") & (df_rel["TEA2"] == "Target")] if not df_rel.empty else pd.DataFrame()

    who = did = what = None
    if not who_did.empty:
        who = who_did.iloc[0]["Node 1"]
        did = who_did.iloc[0]["Node 2"]
    if not did_what.empty:
        if did is None:
            did = did_what.iloc[0]["Node 1"]
        same_verb = did_what[did_what["Node 1"] == did]
        what = same_verb.iloc[0]["Node 2"] if not same_verb.empty else did_what.iloc[0]["Node 2"]

    tea_rows.append({
        "pred_agent": who,
        "pred_event": did,
        "pred_target": what,
        "gold_agent": row["agent"],
        "gold_event": row["event"],
        "gold_target": row["target"],
    })

df_tea_eval = pd.DataFrame(tea_rows)
metrics_tea = {
    "Agent":  accuracy(df_tea_eval["pred_agent"],  df_tea_eval["gold_agent"]),
    "Event":  accuracy(df_tea_eval["pred_event"],  df_tea_eval["gold_event"]),
    "Target": accuracy(df_tea_eval["pred_target"], df_tea_eval["gold_target"]),
}

print("SVO Validation vs. Gold Standard (122 sentences):\n")
print("=== Baseline ===")
df_bl = pd.DataFrame(metrics_baseline).T
display(df_bl)

print("\n=== TEA Nets ===")
df_tea_m = pd.DataFrame(metrics_tea).T
display(df_tea_m)

# --- Show errors ---
def show_errors(df, label):
    print(f"\n{'='*60}")
    print(f"Errors: {label}")
    print(f"{'='*60}")
    for role, pred_col, gold_col in [("Agent", "pred_agent", "gold_agent"),
                                      ("Event", "pred_event", "gold_event"),
                                      ("Target", "pred_target", "gold_target")]:
        p = df[pred_col].apply(norm)
        g = df[gold_col].apply(norm)
        err_mask = p != g
        if err_mask.any():
            print(f"\n--- {role} ({err_mask.sum()} errors) ---")
            for idx in df[err_mask].index:
                sent = str(df.loc[idx, "sentence"])[:60] if "sentence" in df.columns else f"row {idx}"
                print(f"  [{sent}]  pred='{df.loc[idx, pred_col]}'  gold='{df.loc[idx, gold_col]}'")

show_errors(df_eval, "Baseline")
show_errors(df_tea_eval, "TEA Nets")

SVO Validation vs. Gold Standard (122 sentences):

=== Baseline ===


,Correct,Total,Accuracy
Agent,109.0,122.0,0.893
Event,85.0,122.0,0.697
Target,96.0,122.0,0.787



=== TEA Nets ===


,Correct,Total,Accuracy
Agent,111.0,122.0,0.910
Event,99.0,122.0,0.811
Target,105.0,122.0,0.861



Errors: Baseline

--- Agent (13 errors) ---
  [She was given a promotion by her boss.]  pred='boss'  gold='her boss'
  [The software was developed by a small team.]  pred='team'  gold='small team'
  [He was promoted to senior manager.]  pred='to manager'  gold='he'
  [The winner was announced at midnight.]  pred='at midnight'  gold='winner'
  [The data was collected over six months.]  pred='over month'  gold='data'
  [The prisoners were released after the trial.]  pred='after trial'  gold='prisoner'
  [I felt abused by him.]  pred='he'  gold='him'
  [I felt betrayed by my best friend.]  pred='friend'  gold='my good friend'
  [She seemed targeted by everyone.]  pred='she'  gold='everyone'
  [He appeared beaten by the crowd.]  pred='he'  gold='crowd'
  [I got raped by my uncle.]  pred='uncle'  gold='my uncle'
  [She got fired by her boss.]  pred='boss'  gold='her boss'
  [The report was written and reviewed by the committee.]  pred='report'  gold='committee'

--- Event (37 errors) ---
 

In [15]:
from teanets.svo_extraction import _passive_info

PASSIVE_GOLD = "../data/passive_gold.csv"
df_pgold = pd.read_csv(PASSIVE_GOLD)

pred_passive = []
for sent in df_pgold["sentence"].fillna(""):
    doc = nlp(str(sent))
    is_pass = False
    for token in doc:
        if token.pos_ in ("VERB", "AUX") and token.dep_ not in (
            "aux", "auxpass", "amod", "npadvmod", "prep",
        ):
            pinfo = _passive_info(token)
            if pinfo["is_passive"]:
                is_pass = True
                break
    pred_passive.append(int(is_pass))

df_pgold["pred_passive"] = pred_passive

# Overall accuracy
correct_all = (df_pgold["is_passive"] == df_pgold["pred_passive"]).sum()
total_all = len(df_pgold)

# Passive-only accuracy
passive_mask = df_pgold["is_passive"] == 1
correct_passive = (df_pgold.loc[passive_mask, "pred_passive"] == 1).sum()
total_passive = passive_mask.sum()

# Active-only accuracy
active_mask = df_pgold["is_passive"] == 0
correct_active = (df_pgold.loc[active_mask, "pred_passive"] == 0).sum()
total_active = active_mask.sum()

metrics_voice = {
    "Passive": {"Correct": correct_passive, "Total": total_passive, "Accuracy": round(correct_passive / total_passive, 3) if total_passive else 0.0},
    "Active":  {"Correct": correct_active,  "Total": total_active,  "Accuracy": round(correct_active / total_active, 3) if total_active else 0.0},
    "Overall": {"Correct": correct_all,     "Total": total_all,     "Accuracy": round(correct_all / total_all, 3) if total_all else 0.0},
}
display(pd.DataFrame(metrics_voice).T)

# Show mismatches
mismatches = df_pgold[df_pgold["is_passive"] != df_pgold["pred_passive"]]
if not mismatches.empty:
    print(f"\nErrors ({len(mismatches)}):")
    for _, r in mismatches.iterrows():
        pred_label = "passive" if r["pred_passive"] == 1 else "active"
        gold_label = "passive" if r["is_passive"] == 1 else "active"
        print(f"  [{r['sentence'][:70]}]  pred={pred_label}  gold={gold_label}")

,Correct,Total,Accuracy
Passive,206.0,216.0,0.954
Active,911.0,934.0,0.975
Overall,1117.0,1150.0,0.971



Errors (33):
  [An interesting election had occured.]  pred=active  gold=passive
  [I am stunned at the impact politics is having on our country these day]  pred=passive  gold=active
  [Politics have become senseless.]  pred=active  gold=passive
  [Getting involved in politics is easy.]  pred=passive  gold=active
  [There were not enough votes to elect him.]  pred=active  gold=passive
  [Politics can be stressful to be involved in. ]  pred=passive  gold=active
  [I wish there had been different candidates running for President in th]  pred=active  gold=passive
  [Politics never interested me growing up.]  pred=active  gold=passive
  [I think politics are overrated.]  pred=passive  gold=active
  [The senator approval ratings had sunken since voting for the crime bil]  pred=active  gold=passive
  [Gymnastics is one of the sports I like watching. ]  pred=active  gold=passive
  [Sports figures are paid exorbitant amounts of money.]  pred=passive  gold=active
  [The field was filled with p